### Imports

In [11]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import *


from IPython.display import display, clear_output
import pandas as pd 

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


print('librarires imported successfully!!')


librarires imported successfully!!


### Start The Spark Session 

In [15]:
clear_output(wait=True)
# stop spark session is there any 
try : 
    spark.stop()
    print('stopped previous spark session!!')

except : 
    pass 

# start new session 

try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc : 
        sc.stop()
        print('stopped previous sparkContext !!')
except : 
    pass 


spark = SparkSession.builder\
    .appName('ANALYTICAL_FUNCTIONS')\
    .master('spark://spark-master:7077')\
    .config("spark.executor.memory", "3g") \
    .config("spark.driver.memory", "4g") \
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.debug.maxToStringFields', '1000')\
    .getOrCreate()

spark.conf.set('spark.sql.debug.maxToStringFields', '1000')

print('Spark started successfully !!')
print(f'Spark version : {spark.version}')

stopped previous spark session!!
Spark started successfully !!
Spark version : 3.5.1


### Load The Data 

In [16]:
# Define the schema 

fire_schema = StructType([StructField('CallNumber', IntegerType(), True),
StructField('UnitID', StringType(), True),
StructField('IncidentNumber', IntegerType(), True),
StructField('CallType', StringType(), True),                  
StructField('CallDate', StringType(), True),      
StructField('WatchDate', StringType(), True),
StructField('CallFinalDisposition', StringType(), True),
StructField('AvailableDtTm', StringType(), True),
StructField('Address', StringType(), True),       
StructField('City', StringType(), True),       
StructField('Zipcode', IntegerType(), True),       
StructField('Battalion', StringType(), True),                 
StructField('StationArea', StringType(), True),       
StructField('Box', StringType(), True),       
StructField('OriginalPriority', StringType(), True),       
StructField('Priority', StringType(), True),       
StructField('FinalPriority', IntegerType(), True),       
StructField('ALSUnit', BooleanType(), True),       
StructField('CallTypeGroup', StringType(), True),
StructField('NumAlarms', IntegerType(), True),
StructField('UnitType', StringType(), True),
StructField('UnitSequenceInCallDispatch', IntegerType(), True),
StructField('FirePreventionDistrict', StringType(), True),
StructField('SupervisorDistrict', StringType(), True),
StructField('Neighborhood', StringType(), True),
StructField('Location', StringType(), True),
StructField('RowID', StringType(), True),
StructField('Delay', FloatType(), True)])

# Set up the data source file's path 
sf_fire_file = '/opt/spark-data/sf-fire-calls.csv'

# load dataframe with San Francesco fire data 

fire_df = spark.read.csv(sf_fire_file, header= True, schema = fire_schema)
        #.option('samplingRatio', 0.00001)\ # to infere schema

print(' --------- data is loaded successfully --------- ')
# print(f'Number of rows is : {fire_df.count()}')
# print(f'Number of columns is {len(fire_df.columns)}')

 --------- data is loaded successfully --------- 


## 1 Introduction to Window Functions

### 1.1 Windows Types 


In [17]:
# Define the window specification 

# basic window : partition only 
battalion_window = Window.partitionBy('Battalion')

# Ordered window : partition + order 
ranked_window = Window.partitionBy('Battalion').orderBy(desc('score'))

# Frame window : partition + order + frame bound 
moving_avg_window = Window.partitionBy('Neighborhood')\
    .orderBy('week')\
    .rowsBetween(-6, 0)  # Week rolling Window
 

### Example 1: partitionBy

In [22]:
battalion_window = Window.partitionBy('Battalion', 'CallType', year(to_date('CallDate', 'dd/MM/yyyy')))
# add battalion avg to each row 
df_with_avg = fire_df\
    .withColumn('battalion_avg_delay', avg('delay').over(battalion_window))

# calculate difference between battalion average 
df_comparison = df_with_avg\
    .withColumn(
        'vs_battalion',
        col('delay')-col('battalion_avg_delay')
    )

pdf = df_comparison.limit(20).toPandas()
display(pdf)

,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay,battalion_avg_delay,vs_battalion
0,40060177,E13,4001637,Citizen Assist / Service Call,01/06/2004,01/06/2004,Other,01/06/2004 12:37:45 PM,0 Block of BELDEN ST,SF,94104,B01,13,1235,3,3,3,True,None,1,ENGINE,2,1,3,Financial District/South Beach,"(37.7912793098409, -122.403701987134)",040060177-E13,3.033333,6.646667,-3.613333
1,40120243,E01,4003493,Citizen Assist / Service Call,01/12/2004,01/12/2004,Other,01/12/2004 04:05:46 PM,3RD ST/MARKET ST,SF,94103,B01,01,1241,2,2,2,True,None,1,ENGINE,1,3,6,Financial District/South Beach,"(37.7876418476067, -122.403430803415)",040120243-E01,3.683333,6.646667,-2.963333
2,40670131,M01,4019007,Citizen Assist / Service Call,03/07/2004,03/07/2004,Other,03/07/2004 12:15:19 PM,500 Block of TAYLOR ST,SF,94102,B01,03,1451,3,3,3,True,None,1,MEDIC,1,1,3,Tenderloin,"(37.7870711373734, -122.411612352697)",040670131-M01,19.250000,6.646667,12.603333
3,42770238,T16,4076640,Citizen Assist / Service Call,10/03/2004,10/03/2004,Other,10/03/2004 03:58:28 PM,1300 Block of COLUMBUS AVE,SF,94133,B01,A1,1525,3,3,3,False,None,1,TRUCK,2,1,3,North Beach,"(37.8061515935611, -122.418537074676)",042770238-T16,4.066667,6.646667,-2.580000
4,43440277,E03,4094882,Citizen Assist / Service Call,12/09/2004,12/09/2004,Fire,12/09/2004 03:28:55 PM,GEARY ST/TAYLOR ST,SF,94102,B01,03,1451,2,2,2,True,None,1,ENGINE,1,1,3,Tenderloin,"(37.7869397066362, -122.411515768512)",043440277-E03,3.200000,6.646667,-3.446667
5,113090144,T13,11102653,Citizen Assist / Service Call,11/05/2011,11/05/2011,Other,11/05/2011 12:07:40 PM,100 Block of MONTGOMERY ST,SF,94104,B01,13,1235,3,3,3,False,None,1,TRUCK,1,1,3,Financial District/South Beach,"(37.7905126810084, -122.402216710445)",113090144-T13,3.450000,7.338095,-3.888095
6,113390233,E28,11112394,Citizen Assist / Service Call,12/05/2011,12/05/2011,Other,12/05/2011 04:08:50 PM,200 Block of FRANCISCO ST,SF,94133,B01,28,1263,3,3,3,False,None,1,ENGINE,1,1,3,North Beach,"(37.8053066242227, -122.408920785759)",113390233-E28,1.333333,7.338095,-6.004762
7,110050190,RC1,11001640,Citizen Assist / Service Call,01/05/2011,01/05/2011,Other,01/05/2011 12:36:02 PM,1100 Block of JONES ST,SF,94109,B01,41,1465,3,3,3,True,None,1,RESCUE CAPTAIN,2,1,3,Nob Hill,"(37.7918709005093, -122.414268636834)",110050190-RC1,2.500000,7.338095,-4.838095
8,110330172,T13,11010808,Citizen Assist / Service Call,02/02/2011,02/02/2011,Other,02/02/2011 12:21:11 PM,500 Block of BATTERY ST,SF,94111,B01,13,1156,3,3,3,False,None,1,TRUCK,1,1,3,Financial District/South Beach,"(37.7963352483576, -122.400515223296)",110330172-T13,4.666667,7.338095,-2.671429
9,110630422,T03,11020903,Citizen Assist / Service Call,03/04/2011,03/04/2011,Fire,03/04/2011 10:56:22 PM,600 Block of JONES ST,SF,94102,B01,03,1462,3,3,3,False,None,1,TRUCK,1,1,3,Tenderloin,"(37.7870654663911, -122.413159816666)",110630422-T03,5.250000,7.338095,-2.088095


### Example 2: partitionBy(...).orderBy(...)

In [25]:
# Define the rolling window
moving_avg_window = Window\
    .partitionBy('Neighborhood', 'CallType')\
    .orderBy( weekofyear(to_date('CallDate', 'dd/MM/yyyy')))\
    .rowsBetween(-6, 0)


# Calculate the moving average  
df_trend = fire_df\
    .withColumn(
        'moving_avg_7w', 
        avg('delay').over(moving_avg_window)
    )



pdf = df_trend.limit(10).toPandas()
display(pdf)

,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay,moving_avg_7w
0,93210198,E42,9097234,Electrical Hazard,11/17/2009,11/17/2009,Other,11/17/2009 12:42:39 PM,SILVER AV/THOMAS AV,SF,94124,B10,42,6454,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7355808218845, -122.400357336871)",093210198-E42,4.100000,4.100000
1,120820113,E44,12027160,Electrical Hazard,03/22/2012,03/22/2012,Other,03/22/2012 10:28:57 AM,1000 Block of JAMESTOWN AVE,SF,94124,B10,44,6575,3,3,3,True,Alarm,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7189780468693, -122.394528333423)",120820113-E44,2.550000,3.325000
2,180190966,E17,18007998,Electrical Hazard,01/19/2018,01/19/2018,Fire,01/19/2018 09:47:51 AM,2600 Block of ARELIOUS WALKER DR,San Francisco,94124,B10,17,6615,3,3,3,True,Alarm,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.719059825602756, -122.38465573055352)",180190966-E17,4.050000,3.566667
3,102430236,E17,10076675,Electrical Hazard,08/31/2010,08/31/2010,Fire,08/31/2010 02:30:16 PM,0 Block of THORNTON AVE,SF,94124,B10,17,6513,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7307103766598, -122.393023469578)",102430236-E17,3.216667,3.479167
4,53650080,E25,5099259,Electrical Hazard,12/31/2005,12/30/2005,Fire,12/31/2005 05:34:11 AM,DONAHUE ST/INNES AV,SF,94124,B10,25,6663,3,3,3,False,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7288061449325, -122.370145383783)",053650080-E25,7.266667,4.236667
5,60590118,E17,6016438,Electrical Hazard,02/28/2006,02/28/2006,Fire,02/28/2006 09:23:33 AM,100 Block of CAMERON WAY,SF,94124,B10,17,6573,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7209580935201, -122.387166280374)",060590118-E17,4.516667,4.283333
6,83330131,E25,8101870,Electrical Hazard,11/28/2008,11/28/2008,Other,11/28/2008 12:58:05 PM,NEWCOMB AV/PHELPS ST,SF,94124,B10,25,6455,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7373507589088, -122.393523635295)",083330131-E25,4.533333,4.319048
7,91060096,E17,9031146,Electrical Hazard,04/16/2009,04/16/2009,Other,04/16/2009 08:57:21 AM,JENNINGS ST/THOMAS AV,SF,94124,B10,17,6571,3,3,3,False,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.728293726486, -122.387550707572)",091060096-E17,2.400000,4.076191
8,22030041,E17,2060289,Electrical Hazard,07/22/2002,07/21/2002,Other,07/22/2002 05:12:39 AM,INGALLS ST/UNDERWOOD AV,SF,94124,B10,17,6653,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7266061648973, -122.386241894799)",022030041-E17,2.433333,4.059524
9,31360288,B10,3038926,Electrical Hazard,05/16/2003,05/16/2003,Other,05/16/2003 04:51:25 PM,1300 Block of DONNER AVE,SF,94124,B10,17,6615,3,3,3,False,None,1,CHIEF,1,10,10,Bayview Hunters Point,"(37.7216553219361, -122.38929840032)",031360288-B10,12.033334,5.200000


26/10/08 12:56:45 ERROR TaskSchedulerImpl: Lost executor 4 on 172.19.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/10/08 12:56:45 ERROR TaskSchedulerImpl: Lost executor 5 on 172.19.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/10/08 12:56:45 ERROR TaskSchedulerImpl: Lost executor 3 on 172.19.0.5: worker lost: Not receiving heartbeat for 60 seconds


### Example 3: partitionBy().orderBy().rowsBetween( , ) 

In [27]:
# Define window partitioned by Neighborhood, CallType, Week and ordered by Year
yoy_window = Window \
    .partitionBy("Neighborhood", "CallType", weekofyear(to_date('CallDate'))) \
    .orderBy(year(to_date('CallDate')))

# Get previous year's delay
df_yoy = fire_df.withColumn("prev_year_delay", lag("delay", 1).over(yoy_window))

# Calculate YoY growth percentage
df_yoy = df_yoy.withColumn(
    "yoy_growth_pct",
    when(col("prev_year_delay").isNotNull(),
         ((col("delay") - col("prev_year_delay")) / col("prev_year_delay") * 100)
    ).otherwise(None)
)

pdf = df_yoy.limit(20).toPandas()
display(pdf)

,CallNumber,UnitID,IncidentNumber,CallType,CallDate,WatchDate,CallFinalDisposition,AvailableDtTm,Address,City,Zipcode,Battalion,StationArea,Box,OriginalPriority,Priority,FinalPriority,ALSUnit,CallTypeGroup,NumAlarms,UnitType,UnitSequenceInCallDispatch,FirePreventionDistrict,SupervisorDistrict,Neighborhood,Location,RowID,Delay,prev_year_delay,yoy_growth_pct
0,93210198,E42,9097234,Electrical Hazard,11/17/2009,11/17/2009,Other,11/17/2009 12:42:39 PM,SILVER AV/THOMAS AV,SF,94124,B10,42,6454,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7355808218845, -122.400357336871)",093210198-E42,4.100000,NaN,NaN
1,120820113,E44,12027160,Electrical Hazard,03/22/2012,03/22/2012,Other,03/22/2012 10:28:57 AM,1000 Block of JAMESTOWN AVE,SF,94124,B10,44,6575,3,3,3,True,Alarm,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7189780468693, -122.394528333423)",120820113-E44,2.550000,4.100000,-37.804878
2,120930096,E25,12030735,Electrical Hazard,04/02/2012,04/02/2012,Other,04/02/2012 09:57:53 AM,NEWHALL ST/MCKINNON AV,SF,94124,B10,25,6467,3,3,3,True,Alarm,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7369067562737, -122.391087499642)",120930096-E25,3.666667,2.550000,43.790855
3,180190966,E17,18007998,Electrical Hazard,01/19/2018,01/19/2018,Fire,01/19/2018 09:47:51 AM,2600 Block of ARELIOUS WALKER DR,San Francisco,94124,B10,17,6615,3,3,3,True,Alarm,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.719059825602756, -122.38465573055352)",180190966-E17,4.050000,3.666667,10.454548
4,171543967,E42,17065364,Electrical Hazard,06/03/2017,06/03/2017,Fire,06/03/2017 11:04:25 PM,0 Block of BITTING AV,San Francisco,94124,B10,42,6475,3,3,3,True,Alarm,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7269057393186, -122.398315059805)",171543967-E42,8.316667,4.050000,105.349783
5,100610065,E17,10018681,Electrical Hazard,03/02/2010,03/01/2010,Other,03/02/2010 07:17:37 AM,1400 Block of DONNER AVE,SF,94124,B10,17,6615,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7227147456941, -122.391165021387)",100610065-E17,4.583333,8.316667,-44.889777
6,102430236,E17,10076675,Electrical Hazard,08/31/2010,08/31/2010,Fire,08/31/2010 02:30:16 PM,0 Block of THORNTON AVE,SF,94124,B10,17,6513,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7307103766598, -122.393023469578)",102430236-E17,3.216667,4.583333,-29.818184
7,22030041,E17,2060289,Electrical Hazard,07/22/2002,07/21/2002,Other,07/22/2002 05:12:39 AM,INGALLS ST/UNDERWOOD AV,SF,94124,B10,17,6653,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7266061648973, -122.386241894799)",022030041-E17,2.433333,3.216667,-24.352330
8,22850151,E17,2084465,Electrical Hazard,10/12/2002,10/12/2002,Other,10/12/2002 11:53:58 AM,JENNINGS ST/VAN DYKE AV,SF,94124,B10,17,6651,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.7270441476435, -122.388672813188)",022850151-E17,1.850000,2.433333,-23.972604
9,23110014,E17,2091896,Electrical Hazard,11/07/2002,11/06/2002,Other,11/07/2002 02:32:09 AM,3RD ST/WALLACE AV,SF,94124,B10,17,6536,3,3,3,True,None,1,ENGINE,1,10,10,Bayview Hunters Point,"(37.728489580525, -122.392899420759)",023110014-E17,2.000000,1.850000,8.108107


26/10/08 14:46:36 WARN HeartbeatReceiver: Removing executor 7 with no recent heartbeats: 4342684 ms exceeds timeout 120000 ms
26/10/08 14:46:36 WARN HeartbeatReceiver: Removing executor 6 with no recent heartbeats: 4343209 ms exceeds timeout 120000 ms
26/10/08 14:46:36 ERROR TaskSchedulerImpl: Lost executor 7 on 172.19.0.6: Executor heartbeat timed out after 4342684 ms
26/10/08 14:46:36 ERROR TaskSchedulerImpl: Lost executor 6 on 172.19.0.5: Executor heartbeat timed out after 4343209 ms


## 2 Ranking Window Functions :

### 2.1 Basic functions :

In [ ]:

# Define ranking window
ranking_window = Window.partitionBy("District", "CallType") \
    .orderBy(desc("efficiency_score"))

# Apply all four ranking methods
ranked_df = fire_df \
    .withColumn("rank", rank().over(ranking_window)) \
    .withColumn("dense_rank", dense_rank().over(ranking_window)) \
    .withColumn("row_num", row_number().over(ranking_window)) \
    .withColumn("pct_rank", percent_rank().over(ranking_window))

# Filter to top 3 per group
top_3 = ranked_df.filter(col("row_num") <= 3)


### 2.2 Percentile approximation : 

In [30]:

# Calculate multiple percentiles in one aggregation
stats_df = fire_df.groupBy("Neighborhood", "callType").agg(
    expr("percentile_approx(Delay, 0.25)").alias("delay_p25"),   # 25th (Q1)
    expr("percentile_approx(Delay, 0.50)").alias("delay_median"),# 50th (Median)
    expr("percentile_approx(Delay, 0.75)").alias("delay_p75"),   # 75th (Q3)
    expr("percentile_approx(Delay, 0.95)").alias("delay_p95")    # 95th (tail)
)

pdf = stats_df.limit(20).toPandas()
display(pdf)

,Neighborhood,callType,delay_p25,delay_median,delay_p75,delay_p95
0,Bayview Hunters Point,Alarms,2.516667,3.216667,4.183333,5.900000
1,Bayview Hunters Point,Assist Police,74.933334,74.933334,74.933334,74.933334
2,Bayview Hunters Point,Citizen Assist / Service Call,2.750000,3.866667,4.716667,11.516666
3,Bayview Hunters Point,Electrical Hazard,3.150000,4.100000,4.966667,10.216666
4,Bayview Hunters Point,Elevator / Escalator Rescue,2.766667,4.083333,5.250000,5.383333
5,Bayview Hunters Point,Explosion,3.000000,3.016667,4.133333,4.350000
6,Bayview Hunters Point,"Extrication / Entrapped (Machinery, Vehicle)",3.050000,5.200000,6.400000,6.400000
7,Bayview Hunters Point,Fuel Spill,3.183333,4.133333,7.166667,29.266666
8,Bayview Hunters Point,Gas Leak (Natural and LP Gases),3.100000,4.050000,5.233333,11.750000
9,Bayview Hunters Point,HazMat,3.083333,5.316667,6.733333,53.083332


### 2.3 CUBE AND ROLLUP : 

In [31]:

# Rollup creates hierarchical subtotals
rollup_df = fire_df.rollup(
    "SupervisorDistrict",  # Level 1 (outermost)
    "CallTypeGroup",  # Level 2
    "Neighborhood"         # Level 3 (most detailed)
).agg(
    sum("weekly_calls").alias("total_calls"),
    avg("delay").alias("average_delay"),
    count("*").alias("record_count")
)

# Add column to identify aggregation level
rollup_with_level = rollup_df.withColumn("aggregation_level",
    when(col("SupervisorDistrict").isNull(), "GRAND_TOTAL")
    .when(col("call_type_category").isNull(), "DISTRICT_TOTAL")
    .when(col("Neighborhood").isNull(), "CATEGORY_TOTAL")
    .otherwise("NEIGHBORHOOD_DETAIL"))

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `weekly_calls` cannot be resolved. Did you mean one of the following? [`Delay`, `CallDate`, `CallNumber`, `CallType`, `City`].;
'Aggregate [rollup(Vector(0), Vector(1), Vector(2), SupervisorDistrict#135, CallTypeGroup#130, Neighborhood#136)], [SupervisorDistrict#135, CallTypeGroup#130, Neighborhood#136, sum('weekly_calls) AS total_calls#844, avg('avg_delay) AS average_delay#846, count(1) AS record_count#848L]
+- Relation [CallNumber#112,UnitID#113,IncidentNumber#114,CallType#115,CallDate#116,WatchDate#117,CallFinalDisposition#118,AvailableDtTm#119,Address#120,City#121,Zipcode#122,Battalion#123,StationArea#124,Box#125,OriginalPriority#126,Priority#127,FinalPriority#128,ALSUnit#129,CallTypeGroup#130,NumAlarms#131,UnitType#132,UnitSequenceInCallDispatch#133,FirePreventionDistrict#134,SupervisorDistrict#135,Neighborhood#136,Location#137,RowID#138,Delay#139] csv


#### ROLLUP VS CUBE :  
##### ROLLUP:
“Show me totals following a hierarchy”

- Region → Country → City (parent-child relationship)

- Only makes sense to roll up in one direction

##### CUBE:  
“Show me totals for ALL possible combinations”

- City × Product × Year (no natural hierarchy)

- Want to see every slice of the data





![groupby vs rollup vs cube](a.jpeg)


In [ ]:
# Cube creates hierarchical subtotals
rollup_df = fire_df.cube(
    "SupervisorDistrict",  # Level 1 (outermost)
    "CallTypeGroup",  # Level 2
    "Neighborhood"         # Level 3 (most detailed)
).agg(
    sum("weekly_calls").alias("total_calls"),
    avg("delay").alias("average_delay"),
    count("*").alias("record_count")
)

### Stop the Session 

In [32]:
try:
    if spark is not None:
        spark.stop()
        print("Spark session stopped.")
    else:
        print("No Spark session is opened !!")
except NameError:
    print("No Spark session variable defined !!")


Spark session stopped.
